# LAB 01 — Complaint Classifier (LangChain)

Turn a customer complaint into clean, typed data: **category, sentiment, priority**.
This is structured output applied to a real task.

## Setup
Run these two cells first: install the libraries, then create the `llm`.

In [ ]:
# Run once. Installs the workshop libraries.
%pip install -q langchain langchain-core langchain-openai langgraph pydantic

In [ ]:
import os, getpass
from langchain_openai import ChatOpenAI

# Paste your OpenRouter key when prompted (get one at https://openrouter.ai/keys).
if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

# OpenRouter uses the OpenAI API, so we use ChatOpenAI and just change the URL.
# Pass the key explicitly: ChatOpenAI does NOT read OPENROUTER_API_KEY on its own.
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",                       # any model from openrouter.ai/models
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
)
print("LLM ready:", llm.model_name)

## The shape we want back

In [ ]:
from pydantic import BaseModel

class Complaint(BaseModel):
    category: str    # billing, shipping, product, account
    sentiment: str   # angry, neutral, happy
    priority: str    # low, medium, high

classifier = llm.with_structured_output(Complaint)

## Classify some complaints

In [ ]:
complaints = [
    "I've been charged twice this month and nobody is replying to my emails!",
    "The package arrived a day late but everything was fine, just letting you know.",
    "I can't log into my account, the password reset email never arrives.",
]

for text in complaints:
    result = classifier.invoke(f"Classify this customer complaint: {text}")
    print(text)
    print("  ->", result.category, "|", result.sentiment, "|", result.priority)
    print()